In [ ]:
from steer_core.DataManager import DataManager

from steer_materials.CellMaterials.Base import CurrentCollectorMaterial, SeparatorMaterial
from steer_materials.CellMaterials.Electrode import CathodeMaterial, Binder, ConductiveAdditive, AnodeMaterial

from steer_opencell_design.Components.CurrentCollectors import TabWeldedCurrentCollector, WeldTab
from steer_opencell_design.Formulations.ElectrodeFormulations import CathodeFormulation, AnodeFormulation
from steer_opencell_design.Components.Electrodes import Cathode, Anode
from steer_opencell_design.Components.Separators import Separator
from steer_opencell_design.Constructions.Layups.Laminate import Laminate
from steer_opencell_design.Constructions.ElectrodeAssemblies.JellyRolls import WoundJellyRoll
from steer_opencell_design.AuxillaryComponents.WindingEquipment import RoundMandrel

from steer_opencell_design import __version__

import pandas as pd
from datetime import datetime as dt

In [ ]:
db = DataManager()

In [ ]:
db.remove_data(table_name='cells', condition="name = 'Cell 4'")

In [ ]:
db.get_table_names()

In [ ]:
# Get current collector materials from the database

current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
separator_material = SeparatorMaterial.from_database("Polypropylene")

In [ ]:
# Create the cathode

current_collector_tab = WeldTab(
    material=current_collector_material,
    width=10,
    length=100,
    thickness=10
)

cathode_current_collector = TabWeldedCurrentCollector(
    material=current_collector_material,
    weld_tab=current_collector_tab,
    tab_overhang=20,
    skip_coat_width=20,
    length=4000,
    width=120,
    thickness=12,
    weld_tab_positions=[100, 500, 1500, 2500, 3500, 3800],
    tab_weld_side='b'
)

cathode_active_material = CathodeMaterial.from_database("NaNiMn P2-O3 Composite")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=3,
    mass_loading=15,
    insulation_thickness=3
)


In [ ]:
# Create the anode

current_collector_tab = WeldTab(
    material=current_collector_material,
    width=10,
    length=100,
    thickness=10
)

anode_current_collector = TabWeldedCurrentCollector(
    material=current_collector_material,
    weld_tab=current_collector_tab,
    tab_overhang=20,
    skip_coat_width=20,
    length=4010,
    width=123,
    thickness=12,
    weld_tab_positions=[100, 3000],
    tab_weld_side='b'
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=8,
    insulation_thickness=3
)

In [ ]:
# make the layup

top_separator = Separator(
    material=separator_material,
    width=286,
    length=4900,
    thickness=25
)

bottom_separator = Separator(
    material=separator_material,
    width=286,
    length=4900,
    thickness=25
)

my_layup = Laminate(
    anode=my_anode,
    cathode=my_cathode,
    top_separator=top_separator,
    bottom_separator=bottom_separator
)


In [ ]:
# create the jellyroll assembly

mandrel = RoundMandrel(
    diameter=5,
    length=500,
)

my_jellyroll = WoundJellyRoll(
    laminate=my_layup,
    mandrel=mandrel,
    name="Cell 4",
)


In [ ]:
my_jellyroll.roll_properties 

In [ ]:
my_jellyroll.get_spiral_plot(height=1300, width=1300).show() 

In [ ]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_jellyroll.name],
    'object': [my_jellyroll.serialize()],
    'form_factor': ['Cylindrical'],
    'internal_construction': ['Wound'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [ ]:
db.get_data('cells')